In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [2]:
# helper functions
def rotate_3d(theta, axis):
    
    c, s = np.cos(theta), np.sin(theta)

    if axis == 'x':
        return np.array([
            [1, 0, 0],
            [0, c,-s],
            [0, s, c]
        ])
    elif axis == 'y':
        return np.array([
            [c, 0,-s],
            [0, 1, 0],
            [s, 0, c]
        ])

def solve_conic(c, u, v, k, alpha):
    
    cx, cy, cz = c
    ux, uy, uz = u
    vx, vy, vz = v

    A = ux**2 + uy**2 - k**2 * uz**2
    B = 2*ux*vx + 2*uy*vy - 2*k**2 * uz*vz
    C = vx**2 + vy**2 - k**2 * vz**2
    D = 2*cx*ux + 2*cy*uy - 2*k**2 * cz*uz
    E = 2*cx*vx + 2*cy*vy - 2*k**2 * cz*vz
    F = cx**2 + cy**2 - k**2 * cz**2
    
    disc = (B*alpha+E)**2 - 4*C*(A*alpha**2+D*alpha+F)

    nonzero = disc >= 0
    alpha = alpha[nonzero]
    disc = disc[nonzero]

    beta_minus = (-(B*alpha+E) - np.sqrt(disc)) / (2*C)
    beta_plus = (-(B*alpha+E) + np.sqrt(disc)) / (2*C)
    
    return alpha, beta_minus, beta_plus

def build_plane(c, alpha, u, beta, v):
    
    X = c[0] + alpha*u[0] + beta*v[0]
    Y = c[1] + alpha*u[1] + beta*v[1]
    Z = c[2] + alpha*u[2] + beta*v[2]

    return X, Y, Z

def conic_title(cone_angle, plane_angle, epsilon=1e-3):
    if plane_angle < epsilon:
        return 'Circle'
    if plane_angle < cone_angle - epsilon:
        return 'Ellipse'
    elif abs(plane_angle - cone_angle) <= epsilon:
        return 'Parabola'
    else:
        return 'Hyperbola'

In [ ]:
# conic sections animation
resolution = 200
height = 10
scale_factor = 1
frames = 40
centre = np.array([0, 0, -4])
axis = 'x'

plt.style.use('dark_background')
fig = plt.figure(figsize=(10, 5), dpi=200)
fig.suptitle('Conic Sections', fontsize=25)
fig.supxlabel(r'$Ax^2+Bxy+Cy^2+Dx+Ey+F=0$', fontsize=20)
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1], left=0)
ax = [fig.add_subplot(gs[0], projection='3d'), fig.add_subplot(gs[1])]

# plot cone
theta = np.linspace(0, 2*np.pi, resolution)
z = np.linspace(-height, height, resolution)
Theta, Z = np.meshgrid(theta, z)
R = scale_factor * np.abs(Z)
X = R * np.cos(Theta)
Y = R * np.sin(Theta)

ax[0].plot_surface(X, Y, Z, color='red', linewidth=0, alpha=0.5, antialiased=True)

# format plot
lim = scale_factor*height
ax[0].set_aspect('equal', adjustable='box')
ax[0].set_xlim(-lim, lim)
ax[0].set_ylim(-lim, lim)
ax[0].set_zlim(-height, height)

ax[0].xaxis.pane.fill = False
ax[0].yaxis.pane.fill = False
ax[0].zaxis.pane.fill = False
ax[0].xaxis.pane.set_edgecolor('none')
ax[0].yaxis.pane.set_edgecolor('none')
ax[0].zaxis.pane.set_edgecolor('none')
ax[0].xaxis._axinfo['grid']['color'] = (1, 1, 1, 0.3)
ax[0].yaxis._axinfo['grid']['color'] = (1, 1, 1, 0.3)
ax[0].zaxis._axinfo['grid']['color'] = (1, 1, 1, 0.3)
ax[0].set_xticklabels([])
ax[0].set_yticklabels([])
ax[0].set_zticklabels([])

lim_pad = 1.2 * lim
ax[1].set_aspect('equal')
ax[1].set_xlim(-lim_pad, lim_pad)
ax[1].set_ylim(-lim_pad, lim_pad)

ax[1].grid(True, which='major', linestyle='-', linewidth=0.8, alpha=0.3)
ax[1].xaxis.set_major_locator(ticker.MultipleLocator(lim_pad/4))
ax[1].yaxis.set_major_locator(ticker.MultipleLocator(lim_pad/4))
ax[1].set_xticklabels([])
ax[1].set_yticklabels([])

ax[1].set_xlabel('x', fontsize=12)
ax[1].set_ylabel('y', fontsize=12)

curve_title = ax[1].set_title('')

# plot conic section
pad = np.sqrt(2) * max(2*height, 2*scale_factor*height)
initial_alpha = np.linspace(-pad, pad, 5000)

phis = np.linspace(0, 2*np.pi, frames, endpoint=False)

conic_section = ax[0].scatter([], [], [], s=2, color='yellow', axlim_clip=True)
conic_projection = ax[1].scatter([], [], s=12, color='yellow')

def update(frame):

    # calculate plane
    phi = phis[frame]
    rotation = rotate_3d(phi, axis)

    vec_1 = rotation @ np.array([1, 0, 0])
    vec_2 = rotation @ np.array([0, 1, 0])

    alpha, beta_minus, beta_plus = solve_conic(
        centre, vec_1, vec_2, scale_factor, initial_alpha
    )

    x1, y1, z1 = build_plane(centre, alpha, vec_1, beta_minus, vec_2)
    x2, y2, z2 = build_plane(centre, alpha, vec_1, beta_plus, vec_2)
    
    conic_section._offsets3d = (
        np.concatenate([x1, x2]),
        np.concatenate([y1, y2]),
        np.concatenate([z1, z2])
    )

    conic_projection.set_offsets(
        np.column_stack([
            np.concatenate([alpha+centre[0], alpha+centre[0]]),
            np.concatenate([beta_minus+centre[1], beta_plus+centre[1]])
        ])
    )

    normal = np.cross(vec_1, vec_2)
    normal /= np.linalg.norm(normal)
    cone_angle = np.arctan(1/scale_factor)
    plane_angle = np.arccos(abs(normal[2]))
    curve_title.set_text(f'{conic_title(cone_angle, plane_angle)}')

    return conic_section, conic_projection

anim = FuncAnimation(fig, update, frames=frames, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())